# Quantum-Guided Cluster Algorithm for Max-Cut

Use QAOA or PCE two-point correlations to guide classical cluster moves for Max-Cut.

**Reference:** [arXiv:2508.10656](https://arxiv.org/abs/2508.10656)

In [ ]:
import time

from main import run_benchmark, run_multiseed_benchmark

## The Algorithm

The cluster algorithm uses correlations ⟨Z_i Z_j⟩ from a QAOA or PCE run to choose which spins to flip together.

1. **Quantum Phase:** QAOA extracts pairwise correlations ⟨Z_i Z_j⟩, the correlation of spin i and spin j over the measured distribution
2. **Classical Phase:** Cluster Monte Carlo uses these correlations to build spin clusters that are flipped as a unit

### Divi components used

| Feature | Role |
|---------|------|
| **QAOA** on the Max-Cut QUBO | Extracts two-point correlations with `get_correlations()` |
| **QDrift Trotterisation** | Randomised Trotterisation (`QDrift`) |

---

## Local example (10 nodes)

10-node graph, local simulator.

In [ ]:
t0 = time.time()
results_local = run_benchmark(
    n_nodes=10,
    degree=6,
    qaoa_depths=[1, 2],
    n_iterations_factor=200,
    n_repetitions=10,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=5_000,
    output_dir="plots",
)
phase1_time = time.time() - t0
print(f"\nLocal run completed in {phase1_time:.1f}s")

---

## Configuration (28 nodes)

Configuration: **28 nodes, degree 10**, depths p=1, 2, 3, 5. Each QAOA run is dispatched to QoroService, and every run uses **QDrift** trotterisation (randomised Trotter sampling).

**Requirements:**
Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
For a local run, use the smaller configuration in the first example.

In [ ]:
print("Running QAOA on QoroService...")

t0 = time.time()
results_cloud = run_benchmark(
    n_nodes=28,
    degree=10,
    qaoa_depths=[1, 2, 3, 5],
    use_qdrift=True,  # QDrift trotterisation for every depth
    n_iterations_factor=500,
    n_repetitions=30,
    lambda_scale=4,
    seed=42,
    use_cloud=True,
    shots=10_000,
    output_dir="plots",
)
phase2_time = time.time() - t0

print(f"\nLocal  (Phase 1): {phase1_time:.1f}s for 10 nodes")
print(f"Cloud  (Phase 2): {phase2_time:.1f}s for 28 nodes")

## Multi-seed check

Repeats a small local configuration over graph seeds 1, 2 and 3 and prints the mean and range of the QAOA-guided approximation ratios.

In [ ]:
summary = run_multiseed_benchmark(
    seeds=[1, 2, 3],
    n_nodes=8,
    degree=3,
    qaoa_depths=[1],
    n_iterations_factor=10,
    n_repetitions=2,
    lambda_scale=4,
    use_cloud=False,
    shots=1_000,
    output_dir="plots/multiseed",
)
summary

---

## Variation — Swap QAOA for PCE

`run_cluster_algorithm` takes a correlation matrix Z. `extract_qaoa_correlations` and `extract_pce_correlations` return the same `CorrelationResult`; PCE is [Pauli Correlation Encoding](https://arxiv.org/abs/2401.09421).

PCE uses a different encoding of the optimisation variables:

- **Dense:** ⌈log₂(N+1)⌉ qubits
- **Poly:** the smallest q with q(q+1)/2 ≥ N

`run_benchmark` accepts `pce_encodings` alongside `qaoa_depths`, putting both kinds of source on the same comparison plots.

In [ ]:
results_pce = run_benchmark(
    n_nodes=16,
    degree=10,
    qaoa_depths=[2],  # one QAOA depth
    pce_encodings=["dense", "poly"],  # plus both PCE encodings on the same plots
    n_iterations_factor=500,
    n_repetitions=20,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=10_000,
    output_dir="plots_pce",
)

---

## Explore further

Compare QAOA depths, PCE encodings, and cluster-link parameters on a graph size that fits your local environment.